## Module 5: Data Preparation and Training Pipeline

In [1]:
# NaN Problem : Single NaN propagates through every matrix multiplication. The Loss becomes NaN.
import torch
x = torch.tensor([1.0, float('nan'), 3.0])
w = torch.tensor([0.5, 0.5, 0.5])
z = (x * w).sum()
print(z)

tensor(nan)


In [ ]:
# Feature Scaling: we only fit training data to scaler else it will lead to data leakage
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

In [ ]:
# Handling Missing Values

# 1. Deletion
import pandas as pd
df_clean = dp.dropna() # Remove rows with any missing values
df_clean = df.dropna(subset = ['age', 'income']) # Remove rows where specific cols are missing

# 2. Imputation
from sklearn.impute import SimpleImputer
imputer_mean = SimpleImputer(strategy = 'mean') # for num features
imputer_median = SimpleImputer(strategy = 'median') # robust to outliers
imputer_mode = SimpleImputer(strategy = 'most_frequent') # for categorical

imputer_median.fit(x_train)
x_train = imputer_median.transform(x_train)
x_val = imputer_median.transform(x_val)
x_test = imputer_median.transform(x_test)

# 3. Missingness Indicator
df['age_was_missing'] = df['age'].isna().astype(int) # 1 if missing else 0
df['age'] = df['age'].fillna(df['age'].median())

In [ ]:
# Handling Categorical Features

# 1. One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')
encoder.fit(x_train)
x_train = encoder.transform(x_train)
x_val = encoder.transform(x_val)
x_test = encoder.transform(x_test)

# 2. Label Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
encoder = le.fit_transform(y)

In [3]:
# Embeddings for High-Cardinality Categories - for cols where OHE fails due to N number of unique values
import torch.nn as nn
embedding = nn.Embedding(num_embeddings = 1000, embedding_dim = 32)
zip_code_idx = torch.tensor([47, 123, 8, 502])
embedded = embedding(zip_code_idx)
print(embedded.shape)

torch.Size([4, 32])


In [ ]:
# Full Preprocessing Pipeline
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransfromer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_features = ['age', 'income', 'tenure_months', 'montly_charges']
cat_features = ['contract_type', 'payment_method']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])
preprocessor.fit(x_train_df)

x_train = preprocessor.transform(x_train)
x_val = preprocessor.transform(x_val)
x_test = preprocessor.transform(x_test)

import torch
x_train = torch.tensor(x_train, dtype = torch.float32)
x_val = torch.tensor(x_val, dtype = torch.float32)
x_test = torch.tensor(x_test, dtype = torch.float32)

In [5]:
# Splitting Dataset without Data Leakage
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.176, random_state = 42,
                                                 stratify = y_temp)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')
# stratify is used to split imbalanced dataseet propotially.

In [ ]:
# Custom Data Augmentation
class TransfomedDataset(Dataset):
    def __init__(self, X, y, transform = None):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x, y = self.X[idx], self.y[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

def add_noise(x, std = 0.05):  # Adding random gaussian noise during training
    return x + torch.randn_like(x) * std

train_dataset = TransfomedDataset(X_train, y_train, transform = add_noise)
val_dataset = TransfomedDataset(X_val, y_val, transform = none) # no noise for val

In [18]:
# Complete Data Preparation Pipeline: End to End
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# 1. Generating Random Synthetic tabular dataset
np.random.seed(42)
n = 2000
df = pd.DataFrame({
    'age':             np.random.randint(18, 70, n).astype(float),
    'income':          np.random.exponential(50000, n),
    'tenure_months':   np.random.randint(1, 120, n).astype(float),
    'monthly_charges': np.random.uniform(20, 120, n),
    'num_products':    np.random.randint(1, 5, n).astype(float),
    'contract_type':   np.random.choice(['Month-to-month','One year','Two year'], n),
    'payment_method':  np.random.choice(['Auto', 'Manual', 'Electronic'], n)
 })

# Injecting 5% missing values in 'age' and 3% in 'income'
df.loc[np.random.choice(n, int(n * 0.05), replace = False), 'age'] = np.nan
df.loc[np.random.choice(n, int(n * 0.03), replace = False), 'income'] = np.nan

# Binary Target
df['churn'] = (
    ((df['monthly_charges'] > 80) & (df['tenure_months'] < 200)) | (df['contract_type'] == 'month-to-month')
).astype(int)

print(f'Shape: {df.shape}')
print(f'Missing:\n{df.isna().sum()}')
print(f'Class balance: {df["churn"].value_counts(normalize = True).round(3).to_dict()}')


# 2. Splitting the dataset
feature_cols = ['age', 'income', 'tenure_months', 'monthly_charges', 'num_products', 'contract_type', 'payment_method']
X = df[feature_cols]
y = df['churn'].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.176, random_state = 42,
                                                 stratify = y_temp)
print(f'\nSplit: Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')


# 3. Build and fit preprocessing pipeline
num_features = ['age', 'income', 'tenure_months', 'monthly_charges', 'num_products']
cat_features = ['contract_type', 'payment_method']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])
preprocessor.fit(X_train)
X_train = preprocessor.transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

n_features = X_train.shape[1]
print(f'\nFeatures after encoding: {n_features}')


# 4. Create Tensors and Datasets
def make_dataset(X_np, y_np):
    X = torch.tensor(X_np, dtype = torch.float32)
    y = torch.tensor(y_np, dtype = torch.float32).unsqueeze(1)
    return TensorDataset(X, y)

train_ds = make_dataset(X_train, y_train)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)


# 5. DataLoaders
train_loader = DataLoader(train_ds, batch_size = 64, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_ds, batch_size = 128, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 128, shuffle = False, num_workers = 0)


# 6. Model
class ChurnNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16),
            nn.ReLU(), nn.Linear(16, 1)
        )
    
    def forward(self, x):
        return self.net(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChurnNet(in_features = n_features).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
print(f'\nTotal Parameters: {sum(p.numel() for p in model.parameters()):,}')


# 7. Reusable Epoch Runner
def run_epoch(model, loader, criterion, optimizer = None, device = 'cpu'):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for bX, by in loader:
            bX, by = bX.to(device), by.to(device)

            if is_training:
                optimizer.zero_grad()
            out = model(bX)
            loss = criterion(out, by)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * bX.size(0)
            preds = (torch.sigmoid(out) >= 0.5).float()
            correct += (preds == by).sum().item()
            total += bX.size(0)

    return total_loss / total, correct / total * 100


# 8. Training Loop with best-model checkpoint
best_val_loss = float('inf')
history = {'train_loss' : [], 'val_loss' : [], 'train_acc' : [], 'val_acc' : []}

for epoch in range(50):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_churn_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch + 1} | Train: {train_loss:.4f} ({train_acc:.2f}%) |' 
              f'Val {val_loss:.4f} ({val_acc:.2f}%)')


# 9. Final Evaluation on test set
model.load_state_dict(torch.load('best_churn_model.pth'))
test_loss, test_acc = run_epoch(model, test_loader, criterion, None, device)
print(f'\nTest - Loss: {test_loss:.4f} | Accuracy: {test_acc:.2f}%')

# 10. Inference on new customer record
def predict_churn(model, preprocessor, raw_input: dict, device: str) -> dict:
    model.eval()
    input_df = pd.DataFrame([raw_input])
    inputs = preprocessor.transform(input_df)
    inputs = torch.tensor([inputs], dtype = torch.float32 ).to(device)

    with torch.no_grad():
        logit = model(inputs)
        prob = torch.sigmoid(logit).item()

    return {
        'churn probability' : round(prob, 4),
        'predicted_churn' : 1 if prob >= 0.5 else 0,
        'confidence' : 'High' if abs(prob - 0.5) > 0.1 else 'Low'
    }

result = predict_churn(model, preprocessor, {
    'age': 35.0, 'income': 45000.0, 'tenure_months': 8.0,
    'monthly_charges': 95.0, 'num_products': 1.0,
    'contract_type': 'Month-to-month', 'payment_method': 'Manual'}, device)
print(f'\n\nNew Customer Prediction: {result}')

Shape: (2000, 8)
Missing:
age                100
income              60
tenure_months        0
monthly_charges      0
num_products         0
contract_type        0
payment_method       0
churn                0
dtype: int64
Class balance: {0: 0.611, 1: 0.389}

Split: Train: 1400, Val: 300, Test: 300

Features after encoding: 11

Total Parameters: 3,393
Epoch: 10 | Train: 0.0328 (99.21%) |Val 0.0375 (99.00%)
Epoch: 20 | Train: 0.0104 (100.00%) |Val 0.0188 (99.00%)
Epoch: 30 | Train: 0.0047 (100.00%) |Val 0.0085 (100.00%)
Epoch: 40 | Train: 0.0023 (100.00%) |Val 0.0090 (99.67%)
Epoch: 50 | Train: 0.0014 (100.00%) |Val 0.0104 (99.00%)

Test - Loss: 0.0587 | Accuracy: 97.33%


New Customer Prediction: {'churn probability': 1.0, 'predicted_churn': 1, 'confidence': 'High'}


In [19]:
# DataLoader Inspection Utility - to catch data problems before training
def inspect_dataloader(loader, name = 'DataLoader'):
    batch_X, batch_y = next(iter(loader))
    print(f'\n--{name}--')
    print(f"  X shape:       {batch_X.shape}")
    print(f"  y shape:       {batch_y.shape}")
    print(f"  X dtype:       {batch_X.dtype}")
    print(f"  y dtype:       {batch_y.dtype}")
    print(f"  X range:       [{batch_X.min():.3f}, {batch_X.max():.3f}]")
    print(f"  X mean / std:  {batch_X.mean():.3f} / {batch_X.std():.3f}")
    print(f"  NaN in X:      {batch_X.isnan().any().item()}")
    print(f"  NaN in y:      {batch_y.isnan().any().item()}")
    print(f"  Unique labels: {batch_y.unique().tolist()}")
    print(f"  Batches/epoch: {len(loader)}")

inspect_dataloader(train_loader, 'Train Loader')
inspect_dataloader(val_loader, 'Val Loader')


--Train Loader--
  X shape:       torch.Size([64, 11])
  y shape:       torch.Size([64, 1])
  X dtype:       torch.float32
  y dtype:       torch.float32
  X range:       [-1.747, 4.416]
  X mean / std:  0.161 / 0.800
  NaN in X:      False
  NaN in y:      False
  Unique labels: [0.0, 1.0]
  Batches/epoch: 22

--Val Loader--
  X shape:       torch.Size([128, 11])
  y shape:       torch.Size([128, 1])
  X dtype:       torch.float32
  y dtype:       torch.float32
  X range:       [-1.679, 3.387]
  X mean / std:  0.186 / 0.760
  NaN in X:      False
  NaN in y:      False
  Unique labels: [0.0, 1.0]
  Batches/epoch: 3


In [ ]:
# Mini Exercise = Complete Pipeline for California Housing Dataset (regression problem)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

housing = fetch_california_housing()
X, y = housing.data, housing.target
print(f'Features: {housing.feature_names}')
print(f'X: {X.shape}, y range: [{y.min():.2f}, {y.max():.2f}]')

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.176, random_state = 42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_mean, y_std = y_train.mean(), y_train.std()
y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = HousingDataset(X_train, y_train)
val_ds = HousingDataset(X_val, y_val)
test_ds = HousingDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size = 64, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_ds, batch_size = 128, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 128, shuffle = False, num_workers = 0)

bX, by = next(iter(train_loader))
print(f'\nBatch X: {bX.shape}, mean = {bX.mean():.3f}, std = {bX.std():.3f}')
print(f'Batch y: {by.shape}')

class HousingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 32),
            nn.ReLU(), nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = HousingNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

best_val_loss = float('inf')
for epoch in range(50):
    model.train()
    train_loss = 0.0
    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)

        optimizer.zero_grad()
        output = model(bX)
        loss = criterion(output, by)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * bX.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)

            output = model(bX)
            loss = criterion(output, by)

            val_loss += loss.item() * bX.size(0)
    val_loss /= len(val_ds)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_housing.pth')

    if (epoch + 1) % 10 == 0:
        train_rmse = np.sqrt(train_loss) * y_std * 100000
        val_rmse = np.sqrt(val_loss) * y_std * 100000
        print(f'Epoch: {epoch + 1} | Train RMSE: ${train_rmse:.0f} | Val RMSE: ${val_rmse:.0f}')


model.load_state_dict(torch.load('best_housing.pth'))
model.eval()
test_loss = 0.0
all_preds, all_true = [], []

with torch.no_grad():
    for bX, by in test_loader:
        bX, by = bX.to(device), by.to(device)

        output = model(bX)
        loss = criterion(output, by)

        test_loss += loss.item() * bX.size(0)
        all_preds.append(output.cpu())
        all_true.append(by.cpu())

test_loss /= len(test_ds)
test_rmse = np.sqrt(test_loss) * y_std * 100000
pritn(f'\nTest RMSE: {test_rmse:,.0f}')

# Inverse Scaling for Display
all_preds = torch.cat(all_preds).squeeze() * y_std + y_mean
all_true = torch.cat(all_true).squeeze() * y_std + y_mean

print('First 5 Predictions vs True (in $100K):')
for i in range(5):
    print(f'Pred: {all_preds[i]:.3f} | True: {all_true[i]:.3f}')

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X: (20640, 8), y range: [0.15, 5.00]

Batch X: torch.Size([64, 8]), mean = -0.052, std = 0.767
Batch y: torch.Size([64, 1])
Epoch: 10 | Train RMSE: $53760 | Val RMSE: $54411
Epoch: 20 | Train RMSE: $51397 | Val RMSE: $52928
Epoch: 30 | Train RMSE: $49990 | Val RMSE: $52255
Epoch: 40 | Train RMSE: $48498 | Val RMSE: $54790
